# Hash Tables

*Run a cell with ▶ (or Shift+Enter): Sphinx compiles and runs it on your computer. Change the code and run it again: this notebook is your own copy.*

**After this unit you can:**
- Explain how a hash table finds a key in about O(1): hash codes, buckets, collisions and resizing
- Choose a key that makes equal things land together, such as an anagram's sorted letters
- Replace a nested loop with a map of counts or prefix sums
- Build a hash table by hand

**Exercises** in this notebook end with `check(...)` lines: minimal tests that print ✓ when your code is right. Run the next cell once first, so they can use `check`.

In [ ]:
// shared: check(...) prints ✓ or ✗; the exercises below use it
void check(String what, Object actual, Object expected) {
    if (Objects.equals(actual, expected)) {
        IO.println("✓ " + what);
    } else {
        IO.println("✗ " + what + ": expected " + expected + ", got " + actual);
    }
}

## In short

A **hash table** finds a key in about the same time however many keys it holds. A **hash function** turns the key into a number, its **hash code**, and the table uses it to choose a **bucket**, such as `index = hash % buckets`.

![Four keys in five buckets, with a collision in bucket 2](buckets.svg)

Two keys can land in the same bucket: a **collision**. Each bucket keeps a short list (separate chaining), or the table tries the next free slot (open addressing). When the table gets too full it doubles its buckets and reinserts everything, which keeps the lists short: O(1) per operation on average.

#### 🎞️ Putting keys in buckets

![Keys 12, 7, 15 and 9 go to bucket key % 5; 12 and 7 collide in bucket 2, and a lookup only searches that bucket](bucket-steps.svg)

**Collisions happen:** two different strings with the same hash code, and 1,000 words spread over 16 buckets.

In [ ]:
IO.println("\"Aa\".hashCode() = " + "Aa".hashCode());
IO.println("\"BB\".hashCode() = " + "BB".hashCode());

int[] buckets = new int[16];
Random random = new Random(4);
for (int i = 0; i < 1000; i++) {
    String word = "word" + random.nextInt(1_000_000);
    buckets[Math.floorMod(word.hashCode(), 16)]++;
}
IO.println("words per bucket: " + Arrays.toString(buckets));

In Java, `HashMap` and `HashSet` are hash tables. They call the key's `hashCode()` to find the bucket and `equals()` to compare inside it, so the two must agree, and a key must not change while it is in the map.

**equals without hashCode:** the class keeps both "equal" points; the record, which writes both methods for you, keeps one.

In [ ]:
class PointClass {
    final int x;
    final int y;

    PointClass(int x, int y) {
        this.x = x;
        this.y = y;
    }

    @Override
    public boolean equals(Object o) {
        return o instanceof PointClass p && p.x == x && p.y == y;
    }
    // no hashCode: equal points usually land in different buckets
}

record PointRecord(int x, int y) {}

void main() {
    Set<PointClass> classes = new HashSet<>(List.of(new PointClass(1, 2), new PointClass(1, 2)));
    Set<PointRecord> records = new HashSet<>(List.of(new PointRecord(1, 2), new PointRecord(1, 2)));
    IO.println("class without hashCode: " + classes.size() + " points");
    IO.println("record:                 " + records.size() + " point");
}

### ✏️ Exercise: Count the letters

Build a map from each letter to how many times it appears. `merge(key, 1, Integer::sum)` adds one to a count, starting at 1.

In [ ]:
Map<Character, Integer> letterCounts(String word) {
    // TODO: count every char of word in a HashMap
    return new HashMap<>();
}

void main() {
    check("banana", letterCounts("banana"), Map.of('b', 1, 'a', 3, 'n', 2));
    check("empty", letterCounts(""), Map.of());
    check("one letter", letterCounts("zzz"), Map.of('z', 3));
}

<details>
<summary>Show an answer</summary>

```java
Map<Character, Integer> letterCounts(String word) {
    Map<Character, Integer> counts = new HashMap<>();
    for (char c : word.toCharArray()) {
        counts.merge(c, 1, Integer::sum);
    }
    return counts;
}

void main() {
    check("banana", letterCounts("banana"), Map.of('b', 1, 'a', 3, 'n', 2));
    check("empty", letterCounts(""), Map.of());
    check("one letter", letterCounts("zzz"), Map.of('z', 3));
}
```

</details>

Hash tables replace nested loops that ask "have I seen this?" or "how many?":

- **Counting:** `counts.merge(key, 1, Integer::sum)`.
- **Grouping** by a chosen key: words with the same sorted letters are anagrams.
- **Prefix sums:** the sum of `a[i..j]` is `prefix[j + 1] − prefix[i]`, so a map of the prefix sums seen so far finds every range with a given sum in one pass.

**Prefix sums:** after one pass, the sum of any range takes one subtraction.

In [ ]:
int[] a = {3, 4, -1, 2, 5, -3};
long[] prefix = new long[a.length + 1];
for (int i = 0; i < a.length; i++) {
    prefix[i + 1] = prefix[i] + a[i];
}
IO.println("prefix: " + Arrays.toString(prefix));
int[][] ranges = {{0, 2}, {1, 3}, {2, 5}};
for (int[] r : ranges) {
    IO.println("a[" + r[0] + ".." + r[1] + "] sums to " + (prefix[r[1] + 1] - prefix[r[0]]));
}

### ✏️ Exercise: The first repeat

Return the first character that appears for the second time while reading left to right, or `-` if none repeats. A HashSet remembers what you have seen.

In [ ]:
char firstRepeat(String s) {
    // TODO: keep the chars seen so far in a HashSet
    return '-';
}

void main() {
    check("abcab", firstRepeat("abcab"), 'a');
    check("abccba", firstRepeat("abccba"), 'c');
    check("no repeat", firstRepeat("xyz"), '-');
}

<details>
<summary>Show an answer</summary>

```java
char firstRepeat(String s) {
    Set<Character> seen = new HashSet<>();
    for (char c : s.toCharArray()) {
        if (!seen.add(c)) {
            return c;
        }
    }
    return '-';
}

void main() {
    check("abcab", firstRepeat("abcab"), 'a');
    check("abccba", firstRepeat("abccba"), 'c');
    check("no repeat", firstRepeat("xyz"), '-');
}
```

</details>

Watch out: a hash function that sends many keys to one bucket turns the table into a slow list.

**What a bad hash function costs:** the same 10,000 keys, with a good hashCode and with one that returns 42 for everything.

In [ ]:
record GoodKey(int id) {}

record BadKey(int id) {
    @Override
    public int hashCode() {
        return 42; // every key in the same bucket
    }
}

void main() {
    int n = 10_000;
    long start = System.nanoTime();
    Set<GoodKey> good = new HashSet<>();
    for (int i = 0; i < n; i++) {
        good.add(new GoodKey(i));
    }
    long middle = System.nanoTime();
    Set<BadKey> bad = new HashSet<>();
    for (int i = 0; i < n; i++) {
        bad.add(new BadKey(i));
    }
    long end = System.nanoTime();
    IO.println("good hashCode: " + (middle - start) / 1_000_000 + " ms");
    IO.println("hashCode 42:   " + (end - middle) / 1_000_000 + " ms");
}

## Readings

- [Hash Table](https://visualgo.net/en/hashtable) (VisuAlgo (National University of Singapore), 20 min): Insert a few keys with Separate Chaining, then with Linear Probing, and watch where collisions go. Note the load factor α = N/M and why the table is rebuilt bigger when α grows.
- [Prefix Sums](https://usaco.guide/silver/prefix-sums) (USACO Guide, 10 min): The prefix array and the range-sum formula, prefix[r + 1] − prefix[l]. Budget Periods adds one step: a map of the prefix sums seen so far. Pick Java in the site's settings to see Java code.
- [Using Maps to Store Key Value Pairs](https://dev.java/learn/api/collections-and-streams/collections-framework/maps/) (dev.java (Oracle), 7 min): How HashMap stores pairs, the operations to get, check, update and remove them, iterating over the entries, and why a key must not change while it is in a map.

## Check yourself

1. Why does a HashMap stay fast after a million insertions?
2. What key puts "listen", "silent" and "enlist" in the same group?
3. In `3 4 -1 2`, which range sums to 6?

## Practice

The notebook covered the basics. Now apply them in the unit's challenges, in the Sphinx sidebar, where Submit runs every test, hidden ones included:

- **Anagram Groups** · Easy
- **Longest Streak** · Medium
- **Budget Periods** · Medium
- **Hash Table** · Hard
- Test: **Server Log Report** · Hard